# XGBoost sequence classifier

This notebook compares XGBoost with the Random Forest baseline on the engineered sequence features and a stratified 80/20 split. Labels follow the project convention: `0 = Computer`, `1 = Human`; Human is the positive class for reported metrics.

Install the optional dependency if needed: `python -m pip install xgboost`. Run `python DataProcessing/pipeline.py` first to regenerate the feature table.

> Results describe separation from this seeded uniform 1–9 generator and do not establish general computer-generated sequence detection.


## 1. Imports and configuration


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import (
    accuracy_score, classification_report, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

RANDOM_SEED = 42


## 2. Load and validate the feature table

Paths work from either the repository root or the `Models/` directory.


In [ ]:
project_root = Path.cwd()
if not (project_root / "Data").exists():
    project_root = project_root.parent
feature_path = project_root / "Data" / "engineered_features.csv"
if not feature_path.exists():
    raise FileNotFoundError(
        f"{feature_path} not found. Run DataProcessing/pipeline.py from the project root first."
    )

df = pd.read_csv(feature_path)
if "label" not in df.columns:
    raise ValueError("Expected a 'label' column in the engineered feature table.")
if set(df["label"].dropna().unique()) != {0, 1}:
    raise ValueError("Expected labels 0 = Computer and 1 = Human.")
if df.isna().any().any():
    raise ValueError("Feature table contains missing values.")
X = df.drop(columns="label").select_dtypes(include="number")
y = df["label"].astype(int)
print(f"Rows: {len(df):,} | Features: {X.shape[1]}")
display(y.value_counts().sort_index().rename(index={0: "Computer", 1: "Human"}))


## 3. Stratified train/test split

The held-out test partition is not used to fit the model.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED, stratify=y
)
print(f"Training rows: {len(X_train):,} | Test rows: {len(X_test):,}")


## 4. Train XGBoost

The training-only class weight accounts for the smaller Human sample.


In [ ]:
computer_count = int((y_train == 0).sum())
human_count = int((y_train == 1).sum())
model = XGBClassifier(
    n_estimators=500, max_depth=4, learning_rate=0.04,
    subsample=0.85, colsample_bytree=0.85, reg_lambda=1.0,
    scale_pos_weight=computer_count / human_count,
    objective="binary:logistic", eval_metric="logloss",
    random_state=RANDOM_SEED, n_jobs=-1,
)
model.fit(X_train, y_train)


## 5. Held-out evaluation

Review per-class metrics and the confusion matrix along with accuracy. ROC-AUC treats Human (`1`) as positive.


In [ ]:
prediction = model.predict(X_test)
probability_human = model.predict_proba(X_test)[:, list(model.classes_).index(1)]
results = {
    "accuracy": accuracy_score(y_test, prediction),
    "human_precision": precision_score(y_test, prediction, pos_label=1, zero_division=0),
    "human_recall": recall_score(y_test, prediction, pos_label=1, zero_division=0),
    "human_f1": f1_score(y_test, prediction, pos_label=1, zero_division=0),
    "roc_auc_human_positive": roc_auc_score(y_test, probability_human),
}
display(pd.Series(results, name="XGBoost score").round(4))
print(classification_report(
    y_test, prediction, labels=[0, 1],
    target_names=["Computer", "Human"], zero_division=0
))
ConfusionMatrixDisplay.from_predictions(
    y_test, prediction, labels=[0, 1],
    display_labels=["Computer", "Human"], cmap="Blues", values_format="d"
)
plt.title("XGBoost — held-out test set")
plt.tight_layout()
plt.show()


## 6. Feature importance

Tree importance is a model-specific summary, not a causal explanation.


In [ ]:
importance = pd.Series(model.feature_importances_, index=X.columns)
top_importance = importance.nlargest(15).sort_values()
ax = top_importance.plot.barh(figsize=(9, 6), color="#3b82a0")
ax.set_title("XGBoost — top 15 feature importances")
ax.set_xlabel("Gain-based importance")
ax.set_ylabel("")
plt.tight_layout()
plt.show()
display(top_importance.sort_values(ascending=False).rename("importance").to_frame())


## 7. Save the result

Model and report outputs are generated locally and excluded from Git. Re-run this notebook to recreate them.


In [ ]:
import joblib

model_path = project_root / "Models" / "xgboost_model.joblib"
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump({
    "model": model,
    "feature_columns": list(X.columns),
    "label_names": {0: "Computer", 1: "Human"},
}, model_path)
report = {
    "model": "XGBoost", "seed": RANDOM_SEED,
    "train_rows": len(X_train), "test_rows": len(X_test),
    "feature_count": X.shape[1],
    "label_mapping": {"0": "Computer", "1": "Human"},
    "metrics": {key: float(value) for key, value in results.items()},
    "interpretation_limit": (
        "Performance describes the observed human sample versus the seeded uniform 1-9 generator; "
        "it does not establish general computer-generated sequence detection."
    ),
}
report_dir = project_root / "Reports"
report_dir.mkdir(parents=True, exist_ok=True)
(report_dir / "xgboost_metrics.json").write_text(
    json.dumps(report, indent=2), encoding="utf-8"
)
print(f"Saved model: {model_path}")
print(f"Saved metrics: {report_dir / 'xgboost_metrics.json'}")
